In [1]:
import pandas as pd
import polars as pl

In [2]:
def add_finish_reason(df: pl.DataFrame) -> pl.DataFrame:
    """
    Add a finish_reason column to the dataframe.
    If the response ends with "</s>", the finish_reason is "stop", otherwise "length".
    """
    df = df.with_columns(
        pl.when(pl.col("response").str.ends_with("</s>"))
        .then(pl.lit("stop"))
        .otherwise(pl.lit("length"))
        .alias("stop_reason")
    )
    return df

In [3]:
# cleanup response column
def cleanup_response(df: pl.DataFrame) -> pl.DataFrame:
    """
    Cleanup the response column by removing unwanted tokens.
    """
    system_prompt = "<|im_start|> system\nYou are a helpful assistant. Please give short and concise answers.<|im_end|> \n<|im_start|> user\n"
    replace_dict = {
        system_prompt: "",
        "<|im_end|>": "",
        "</s>": "",
        "<s>": "",
        "<|im_start|> user\n": "",
    }
    df = df.with_columns(
        pl.col("response")
        .str.replace(system_prompt, "")
        .str.replace_many(replace_dict)
        .alias("response")
    )
    return df


In [4]:
def get_model_response(df: pl.DataFrame) -> pl.DataFrame:
    """
    Extract the model response from the response column.
    Only keep text after "assistant\n".
    """
    df = df.with_columns(
        pl.col("response")
        .str.split("assistant\n") # split by "assistant\n"
        .list.get(-1) # get last element
        .alias("output")
    )
    return df

# German

In [5]:
df = pd.read_json("output/occiglot-7b-eu5-instruct_de.jsonl", lines=True)
df = pl.from_pandas(df)

In [6]:
df = add_finish_reason(df)
df = cleanup_response(df)
df = get_model_response(df)

In [7]:
df = df.with_columns(
    pl.col("id").alias("prompt_id"),
    pl.col("timestamp").alias("output_id"),
).select([
    "prompt_id",
    "output_id",
    "output",
    "stop_reason"
])

In [8]:
df = df.with_columns(
    pl.lit("occiglot-7b-eu5-instruct").alias("model")
)

In [ ]:
# cleanup instances with only whitespaces and newlines
df = df.with_columns(
    pl.when(pl.col("output").str.strip_chars().eq(""))
    .then(pl.lit(""))
    .otherwise(pl.col("output"))
    .alias("output")
)

In [10]:
df.write_csv("output/occiglot-7b-eu5-instruct_de.csv")

# French

In [11]:
df = pd.read_json("output/occiglot-7b-eu5-instruct_fr.jsonl", lines=True)
df = pl.from_pandas(df)

In [12]:
df = add_finish_reason(df)
df = cleanup_response(df)
df = get_model_response(df)
df = df.with_columns(
    pl.col("id").alias("prompt_id"),
    pl.col("timestamp").alias("output_id")
).select([
    "prompt_id",
    "output_id",
    "output",
    "stop_reason"
])
df = df.with_columns(
    pl.lit("occiglot-7b-eu5-instruct").alias("model")
)

In [13]:
df.write_csv("output/occiglot-7b-eu5-instruct_fr.csv")

# Italian

In [14]:
df = pd.read_json("output/occiglot-7b-eu5-instruct_it.jsonl", lines=True)
df = pl.from_pandas(df)

In [15]:
df = add_finish_reason(df)
df = cleanup_response(df)
df = get_model_response(df)
df = df.with_columns(
    pl.col("id").alias("prompt_id"),
    pl.col("timestamp").alias("output_id")
).select([
    "prompt_id",
    "output_id",
    "output",
    "stop_reason"
])
df = df.with_columns(
    pl.lit("occiglot-7b-eu5-instruct").alias("model")
)

In [16]:
df.write_csv("output/occiglot-7b-eu5-instruct_it.csv")